# STAGE 1 – SINH 3 CỘT SỐ BẰNG `np.random` THEO QUY LUẬT

Thay thế bước `AI_tao_sinh_2` (Ollama). Notebook này **chỉ cần** `cgv_showtimes_AI_generated.csv` (lịch chiếu + `GeneratedText` từ `AI_tao_sinh`) và tạo thêm 3 cột `TicketPrice`, `TotalSeats`, `AvailableSeats` bằng `numpy`.

In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

df = pd.read_csv("cgv_showtimes_AI_generated.csv", encoding="utf-8-sig")
print(df.shape, df.columns.tolist())
df.head()

(1172, 6) ['MovieName', 'Cinema', 'ShowDate', 'Format', 'ShowTime', 'GeneratedText']


,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI ...
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải n..."
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI ...
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Cr..."
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN..."


## 1. Đặc trưng suy ra từ chính lịch chiếu

Chỉ dùng các cột đã cào được: ngày chiếu, giờ chiếu, định dạng phòng, tên phim.

In [2]:
date = pd.to_datetime(df["ShowDate"], format="%d/%m/%Y")
hour = pd.to_datetime(df["ShowTime"], format="%H:%M").dt.hour
fmt = df["Format"].str.upper()

is_weekend = date.dt.dayofweek.isin([4, 5, 6]).astype(int)   # thứ 6, 7, CN
is_prime   = hour.between(18, 21).astype(int)                # giờ vàng 18-21h
is_morning = (hour < 12).astype(int)
is_late    = (hour >= 22).astype(int)
is_premium = fmt.str.contains("GOLD CLASS|CINE|LIVING|IMAX|4DX|SCREENX|STARIUM").astype(int)

top_movies = df["MovieName"].value_counts().index[:3]        # 3 phim nhiều suất nhất = phim hot
is_hot = df["MovieName"].isin(top_movies).astype(int)
print("Phim hot:", list(top_movies))

Phim hot: ['TRẠI BUÔN NGƯỜI', 'AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI)', 'VÙNG ĐẤT QUỶ DỮ']


## 2. Quy luật giả định

| Cột | Quy luật |
|---|---|
| `TotalSeats` | Phòng cao cấp (GOLD CLASS, Cine & Living Room) = 80 ghế; IMAX/4DX/ScreenX = 120 ghế; còn lại = 100 ghế; 5% ngoại lệ ngẫu nhiên |
| `TicketPrice` | Nền 70.000 + phòng cao cấp + cuối tuần + giờ vàng + phòng 120 ghế − suất sáng + nhiễu, làm tròn 5.000 |
| `AvailableSeats` | Tỉ lệ lấp đầy tăng với phim hot, cuối tuần, giờ vàng; giảm với suất sáng, suất khuya, giá vé cao; số ghế bán ~ nhị thức |

In [3]:
# --- TotalSeats ---
total = np.where(fmt.str.contains("GOLD CLASS|CINE|LIVING"), 80,
         np.where(fmt.str.contains("IMAX|4DX|SCREENX"), 120, 100))
noise_mask = rng.random(len(df)) < 0.05
total = np.where(noise_mask, rng.choice([80, 100, 120], len(df)), total).astype(int)

# --- TicketPrice ---
price = (70000
         + 25000 * is_premium
         + 8000 * is_weekend
         + 7000 * is_prime
         - 5000 * is_morning
         + 5000 * (total == 120)
         + rng.normal(0, 2500, len(df)))
price = (np.round(price / 5000) * 5000).clip(50000, 130000).astype(int)

# --- AvailableSeats ---
occ = (0.12
       + 0.20 * is_hot
       + 0.10 * is_weekend
       + 0.12 * is_prime
       - 0.06 * is_morning
       - 0.07 * is_late
       - 0.10 * (price - 90000) / 30000
       + rng.normal(0, 0.05, len(df)))
occ = np.clip(occ, 0.02, 0.95)
sold = rng.binomial(total, occ)
available = (total - sold).astype(int)

## 3. Gắn 3 cột vào lịch chiếu và kiểm tra ràng buộc

In [4]:
df_v2 = df.copy()
df_v2["TicketPrice"] = price
df_v2["AvailableSeats"] = available
df_v2["TotalSeats"] = total

print("Thiếu giá trị:", df_v2[["TicketPrice", "AvailableSeats", "TotalSeats"]].isna().sum().to_dict())
print("TicketPrice trong 50.000–130.000:", df_v2["TicketPrice"].between(50000, 130000).all())
print("TotalSeats thuộc {80,100,120}:", df_v2["TotalSeats"].isin([80, 100, 120]).all())
print("0 <= AvailableSeats <= TotalSeats:",
      ((df_v2["AvailableSeats"] >= 0) & (df_v2["AvailableSeats"] <= df_v2["TotalSeats"])).all())
print("Tỉ lệ lấp đầy trung bình:", round((1 - df_v2["AvailableSeats"] / df_v2["TotalSeats"]).mean(), 3))
df_v2[["TicketPrice", "AvailableSeats", "TotalSeats"]].describe().round(1)

Thiếu giá trị: {'TicketPrice': 0, 'AvailableSeats': 0, 'TotalSeats': 0}
TicketPrice trong 50.000–130.000: True
TotalSeats thuộc {80,100,120}: True
0 <= AvailableSeats <= TotalSeats: True
Tỉ lệ lấp đầy trung bình: 0.336


,TicketPrice,AvailableSeats,TotalSeats
count,1172.0,1172.0,1172.0
mean,79709.9,67.1,100.9
std,12230.1,15.6,8.6
min,60000.0,26.0,80.0
25%,70000.0,56.0,100.0
50%,75000.0,66.0,100.0
75%,85000.0,78.0,100.0
max,120000.0,119.0,120.0


In [5]:
df_v2.to_csv("cgv_showtimes_AI_generated_v2.csv", index=False, encoding="utf-8-sig")
print("Đã lưu cgv_showtimes_AI_generated_v2.csv", df_v2.shape)

Đã lưu cgv_showtimes_AI_generated_v2.csv (1172, 9)
